# 🧪 MST-GNN Pipeline Smoke Test

**Mục đích:** Test toàn bộ pipeline nhanh (~5 min) để phát hiện bugs trước khi chạy file chính.

| Phase | Test | Expected |
|-------|------|----------|
| Setup | GPU + Clone + Install | ✅ |
| Phase 1 | Data loading | DataFrame có rows |
| Phase 2 | Preprocessing | Features + sliding windows |
| Phase 3 | Graph building | List of DGL graphs |
| Phase 4 | Dataset creation | Train/Val/Test splits |
| Phase 5 | Training (3 epochs) | Loss decreasing |
| Phase 6 | Backtest | Cumulative returns plot |
| Git | Auto-push | Commit + Push thành công |

**Giới hạn:** CSI300 data, 6 tháng (2025-01 → 2025-06), chỉ 3 epochs.


---
## Cell 1 — Setup


In [ ]:
import torch, subprocess, os, sys, time, traceback

print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

REPO_URL = "https://github.com/quocnguyen5/mst-gnn-impl.git"
WORK_DIR = "/kaggle/working/mst-gnn-impl"

if os.path.exists(WORK_DIR):
    !cd {WORK_DIR} && git pull --rebase
else:
    !git clone {REPO_URL} {WORK_DIR}

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

!pip install -q torch_geometric akshare vnstock jieba pyarrow fastparquet tqdm 2>&1 | tail -3

# Git token (optional for test)
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    os.environ["GITHUB_TOKEN"] = token
    !git remote set-url origin https://{token}@github.com/quocnguyen5/mst-gnn-impl.git
    !git config user.email "nguyennq.app@gmail.com"
    !git config user.name "Kaggle AutoPush"
    print("✅ GitHub token configured.")
except:
    os.environ["GITHUB_TOKEN"] = ""
    print("⚠️  No GitHub token — git push test will be skipped.")

print("\n✅ Setup complete!")


---
## Cell 2 — Full Pipeline Smoke Test (CSI300, 6 months, 3 epochs)


In [ ]:
import time, traceback, os, sys, json
import numpy as np

T_TOTAL = time.time()
results = {}
PASS = "✅ PASS"
FAIL = "❌ FAIL"

# ════════════════════════════════════════════════
# Phase 1: Data Loading
# ════════════════════════════════════════════════
print("=" * 60)
print("  PHASE 1: Data Loading")
print("=" * 60)
try:
    t = time.time()
    from config import Config
    from data.collector import StockDataCollector

    config = Config.for_csi300()

    collector = StockDataCollector(cache_dir=config.data.raw_data_dir)
    raw_data = collector.collect_all(dataset="csi300")

    prices = raw_data["daily_prices"]
    industry = raw_data["industry"]
    news = raw_data["news"]

    # ★ LIMIT to 6 months for speed
    import pandas as pd
    prices["date"] = pd.to_datetime(prices["date"])
    prices = prices[prices["date"] >= "2025-01-01"]
    prices = prices[prices["date"] <= "2025-06-30"]
    n_stocks = prices["stock_code"].nunique()
    n_rows = len(prices)

    assert n_rows > 0, "No price rows after date filter!"
    assert n_stocks > 10, f"Only {n_stocks} stocks!"

    print(f"  Prices:   {n_rows:,} rows, {n_stocks} stocks")
    print(f"  Industry: {len(industry)} records")
    print(f"  News:     {len(news)} records")
    print(f"  Date range: {prices['date'].min().date()} → {prices['date'].max().date()}")
    print(f"  Time: {time.time()-t:.1f}s")
    results["Phase 1: Data Loading"] = PASS
except Exception as e:
    print(f"  ERROR: {e}")
    traceback.print_exc()
    results["Phase 1: Data Loading"] = FAIL

# ════════════════════════════════════════════════
# Phase 2: Preprocessing
# ════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 2: Preprocessing")
print("=" * 60)
try:
    t = time.time()
    from data.preprocessing import Preprocessor

    preprocessor = Preprocessor(
        lookback=config.data.lookback,
        horizon=config.data.horizon,
    )
    trading_dates, samples = preprocessor.create_samples(prices)
    n_dates = len(trading_dates)
    n_samples = sum(len(s) for s in samples.values()) if isinstance(samples, dict) else len(samples)

    assert n_dates > 10, f"Only {n_dates} trading dates!"
    print(f"  Trading dates: {n_dates}")
    print(f"  Samples: {n_samples:,}")
    print(f"  Date range: {trading_dates[0]} → {trading_dates[-1]}")
    print(f"  Time: {time.time()-t:.1f}s")
    results["Phase 2: Preprocessing"] = PASS
except Exception as e:
    print(f"  ERROR: {e}")
    traceback.print_exc()
    results["Phase 2: Preprocessing"] = FAIL

# ════════════════════════════════════════════════
# Phase 3: Graph Building (NO CACHE — fresh build)
# ════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 3: Graph Building (small dataset, ~2 min)")
print("=" * 60)
try:
    t = time.time()
    from data.graph_builder import GraphBuilder

    stock_codes = sorted(prices["stock_code"].unique().tolist())
    graph_builder = GraphBuilder(
        num_topics=config.model.num_topics,
        num_stocks=len(stock_codes),
    )

    graphs = graph_builder.build_all_graphs(
        prices_df=prices,
        industry_df=industry,
        shareholding_df=raw_data.get("shareholding", pd.DataFrame()),
        news_df=news,
        stock_codes=stock_codes,
        trading_dates=trading_dates,
    )

    assert len(graphs) > 0, "No graphs built!"
    print(f"  Graphs built: {len(graphs)}")
    # Check first graph structure
    g0 = list(graphs.values())[0] if isinstance(graphs, dict) else graphs[0]
    print(f"  First graph: {g0}")
    print(f"  Time: {(time.time()-t)/60:.1f} min")
    results["Phase 3: Graph Building"] = PASS
except Exception as e:
    print(f"  ERROR: {e}")
    traceback.print_exc()
    results["Phase 3: Graph Building"] = FAIL

# ════════════════════════════════════════════════
# Phase 4: Dataset Creation
# ════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 4: Dataset Creation")
print("=" * 60)
try:
    t = time.time()
    from data.dataset import DatasetBuilder

    dataset_builder = DatasetBuilder(
        train_ratio=config.data.train_ratio,
        val_ratio=config.data.val_ratio,
        test_ratio=config.data.test_ratio,
        cache_dir=config.data.processed_data_dir,
    )

    snapshots = dataset_builder.build_snapshots(trading_dates, samples, graphs)
    train_ds, val_ds, test_ds = dataset_builder.split_dataset(snapshots)

    assert len(train_ds) > 0, "Empty train set!"
    assert len(val_ds) > 0, "Empty val set!"
    assert len(test_ds) > 0, "Empty test set!"
    print(f"  Total:  {len(snapshots)} snapshots")
    print(f"  Train:  {len(train_ds)}")
    print(f"  Val:    {len(val_ds)}")
    print(f"  Test:   {len(test_ds)}")
    print(f"  Time: {time.time()-t:.1f}s")
    results["Phase 4: Dataset Creation"] = PASS
except Exception as e:
    print(f"  ERROR: {e}")
    traceback.print_exc()
    results["Phase 4: Dataset Creation"] = FAIL

# ════════════════════════════════════════════════
# Phase 5: Training (3 epochs only!)
# ════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 5: Training (3 epochs — smoke test)")
print("=" * 60)
try:
    t = time.time()
    from models.mst_gnn import MSTGNN
    from train import Trainer

    # Override to 3 epochs, no early stopping
    config.train.num_epochs = 3
    config.train.patience = 999  # disable early stopping
    config.train.checkpoint_interval = 1  # checkpoint every epoch
    config.train.experiment_name = "smoke_test"
    os.makedirs(config.train.save_dir, exist_ok=True)

    model = MSTGNN.from_config(config)
    n_params = model.count_parameters()
    print(f"  Model params: {n_params:,}")

    trainer = Trainer(
        model=model,
        config=config,
        train_dataset=train_ds,
        val_dataset=val_ds,
        test_dataset=test_ds,
    )

    test_metrics = trainer.train(auto_resume=False)  # fresh start

    assert "accuracy" in test_metrics, "No accuracy in test_metrics!"
    assert "damrr" in test_metrics, "No damrr in test_metrics!"
    print(f"\n  Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"  Test DAMRR:    {test_metrics['damrr']:.4f}")
    print(f"  Test Loss:     {test_metrics['loss']:.4f}")
    print(f"  Best Epoch:    {trainer.best_epoch}")
    print(f"  Time: {(time.time()-t)/60:.1f} min")
    results["Phase 5: Training"] = PASS
except Exception as e:
    print(f"  ERROR: {e}")
    traceback.print_exc()
    results["Phase 5: Training"] = FAIL

# ════════════════════════════════════════════════
# Phase 6: Backtest
# ════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 6: Trading Simulation / Backtest")
print("=" * 60)
try:
    t = time.time()
    from evaluate import TradingSimulator

    dates, codes, preds, scores, returns = trainer.get_predictions(test_ds)

    simulator = TradingSimulator(
        top_k_stocks=config.backtest.top_k_stocks,
        transaction_cost=config.backtest.transaction_cost,
    )
    backtest_results = simulator.simulate(dates, codes, scores, returns)
    report = simulator.generate_report(backtest_results)
    print(report.to_string(index=False))

    save_path = os.path.join(config.train.save_dir, "cumulative_returns_smoke_test.png")
    simulator.plot_results(backtest_results, dates=dates, save_path=save_path)
    print(f"  Chart saved: {save_path}")
    print(f"  Time: {time.time()-t:.1f}s")
    results["Phase 6: Backtest"] = PASS
except Exception as e:
    print(f"  ERROR: {e}")
    traceback.print_exc()
    results["Phase 6: Backtest"] = FAIL

# ════════════════════════════════════════════════
# Phase 7: JSON Save + Checkpoint Verify
# ════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 7: Results Persistence")
print("=" * 60)
try:
    import glob
    # Check best model checkpoint
    best_ckpt = f"checkpoints/best_model_smoke_test.pt"
    if os.path.exists(best_ckpt):
        ckpt = torch.load(best_ckpt, map_location="cpu", weights_only=False)
        has_history = "train_history" in ckpt and len(ckpt["train_history"]) > 0
        print(f"  best_model: ✅ (epoch {ckpt['epoch']}, history={has_history})")
    else:
        print(f"  best_model: ❌ NOT FOUND at {best_ckpt}")
        # Check what exists
        for f in glob.glob("checkpoints/*.pt"):
            print(f"    Found: {f}")

    # Save test results JSON
    results_json = {
        "dataset": "smoke_test",
        "test_metrics": {k: round(v, 4) for k, v in test_metrics.items()},
        "best_epoch": trainer.best_epoch,
        "total_epochs": len(trainer.train_history),
        "train_history": trainer.train_history,
        "val_history": trainer.val_history,
    }
    json_path = "checkpoints/results_smoke_test.json"
    with open(json_path, "w") as f:
        json.dump(results_json, f, indent=2)
    print(f"  results JSON: ✅ saved to {json_path}")
    results["Phase 7: Persistence"] = PASS
except Exception as e:
    print(f"  ERROR: {e}")
    traceback.print_exc()
    results["Phase 7: Persistence"] = FAIL

# ════════════════════════════════════════════════
# Phase 8: Git Push Test
# ════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  PHASE 8: Git Push")
print("=" * 60)
try:
    token = os.environ.get("GITHUB_TOKEN", "")
    if token:
        def _run(cmd):
            r = subprocess.run(cmd, capture_output=True, text=True, cwd=WORK_DIR, timeout=120)
            return r.returncode, r.stdout + r.stderr

        _run(["git", "add", "-A"])
        rc, out = _run(["git", "commit", "-m", "Smoke test results"])
        if rc != 0 and "nothing to commit" in out:
            print("  Nothing to commit.")
        else:
            rc, out = _run(["git", "push"])
            if rc != 0:
                _run(["git", "pull", "--rebase", "--no-edit"])
                rc, out = _run(["git", "push"])
            if rc != 0:
                rc, out = _run(["git", "push", "--force"])

            if rc == 0:
                print("  ✅ Git push successful!")
            else:
                print(f"  ❌ Git push failed: {out[:200]}")
        results["Phase 8: Git Push"] = PASS
    else:
        print("  ⏭️  Skipped (no GITHUB_TOKEN)")
        results["Phase 8: Git Push"] = "⏭️ SKIP"
except Exception as e:
    print(f"  ERROR: {e}")
    results["Phase 8: Git Push"] = FAIL

# ════════════════════════════════════════════════
# FINAL REPORT
# ════════════════════════════════════════════════
total_min = (time.time() - T_TOTAL) / 60
print("\n\n" + "═" * 60)
print("  🧪 SMOKE TEST REPORT")
print("═" * 60)
all_pass = True
for phase, status in results.items():
    print(f"  {status}  {phase}")
    if "FAIL" in status:
        all_pass = False
print(f"\n  Total time: {total_min:.1f} min")
if all_pass:
    print("\n  🎉 ALL TESTS PASSED — Ready to run full experiments!")
else:
    print("\n  ⚠️  Some tests failed — fix issues before running full experiments.")
print("═" * 60)


---
## Cell 3 — Backtest Chart


In [ ]:
from IPython.display import Image as IPImage, display
chart = "checkpoints/cumulative_returns_smoke_test.png"
if os.path.exists(chart):
    display(IPImage(filename=chart))
else:
    print("Chart not generated — check Phase 6 above.")
